In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 00:03:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Green

In [2]:
df_green = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("../../data/raw/green")
)  # read recursively 

In [3]:
df_green.columns

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [4]:
df_green.registerTempTable("green")

/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/.venv/lib/python3.10/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [5]:
df_green_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    count(1) as number_records   
FROM
    green
where lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
-- order by 
--     1, 2
""")

# spark master showed 2 blockes of tasks for only group by , and 3 blocks of tasks for group by and order by 

In [6]:
df_green_revenue.show()

[Stage 1:========>                                                (2 + 12) / 14]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00| 244|183.57999999999998|            12|
|2020-01-01 00:00:00|  71|              23.8|             1|
|2020-01-01 02:00:00| 179|            206.82|            11|
|2020-01-01 02:00:00| 112|            417.35|            20|
|2020-01-01 04:00:00|  93|184.66000000000003|             2|
|2020-01-01 06:00:00| 193|              16.8|             1|
|2020-01-01 10:00:00|  25|             16.62|             1|
|2020-01-01 14:00:00| 196|110.88000000000001|             7|
|2020-01-02 06:00:00| 175|              37.0|             1|
|2020-01-02 10:00:00| 190|172.20000000000002|             3|
|2020-01-02 11:00:00|  98|               9.3|             1|
|2020-01-02 15:00:00| 146|            130.29|             3|
|2020-01-02 17:00:00| 171|              22.8|             1|
|2020-01-02 18:00:00|  5

In [7]:
df_green_revenue.write.parquet("../../data/report/revenue/green", mode="overwrite")

26/03/10 00:03:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/10 00:03:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/10 00:03:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/10 00:03:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/10 00:03:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/10 00:03:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/10 00:03:13 WARN MemoryManager: Total allocation exceeds 95.

# Yellow

In [8]:
df_yellow = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("../../data/raw/yellow")
)  # read recursively 

df_yellow.registerTempTable("yellow")

df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee']

In [9]:
df_yellow_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    count(1) as number_records   
FROM
    yellow
where tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
-- order by 
--     1, 2
""")

# spark master showed 2 blockes of tasks for only group by , and 3 blocks of tasks for group by and order by 

df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00| 244|183.57999999999998|            12|
|2020-01-01 00:00:00|  71|              23.8|             1|
|2020-01-01 02:00:00| 179|            206.82|            11|
|2020-01-01 02:00:00| 112|            417.35|            20|
|2020-01-01 04:00:00|  93|184.66000000000003|             2|
|2020-01-01 06:00:00| 193|              16.8|             1|
|2020-01-01 10:00:00|  25|             16.62|             1|
|2020-01-01 14:00:00| 196|110.88000000000001|             7|
|2020-01-02 06:00:00| 175|              37.0|             1|
|2020-01-02 10:00:00| 190|172.20000000000002|             3|
|2020-01-02 11:00:00|  98|               9.3|             1|
|2020-01-02 15:00:00| 146|            130.29|             3|
|2020-01-02 17:00:00| 171|              22.8|             1|
|2020-01-02 18:00:00|  5

In [10]:
df_yellow_revenue.write.parquet("../../data/report/revenue/yellow", mode="overwrite")
# alternatively (even though in practise it is ok if we have like 200 partitions if files are large 
# but here we could also simplify: 
# df_yellow_revenue.repartition(20).write.parquet("../../data/report/revenue/yellow", mode="overwrite")

26/03/10 00:03:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/10 00:03:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/10 00:03:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/10 00:03:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/10 00:03:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/10 00:03:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/10 00:03:18 WARN MemoryManager: Total allocation exceeds 95.

In [11]:
# now i want these two tables into one table

In [18]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [19]:
df_join.show()

[Stage 32:=====================================================>  (19 + 1) / 20]

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  34|              NULL|                NULL|              19.3|                    1|
|2020-01-01 00:00:00|  41| 1363.959999999999|                  84|1256.5299999999988|                   80|
|2020-01-01 00:00:00|  71|              23.8|                   1|              NULL|                 NULL|
|2020-01-01 00:00:00|  87|              NULL|                NULL|           2456.67|                  112|
|2020-01-01 00:00:00| 112|312.26000000000005|                  18|119.47999999999999|                    8|
|2020-01-01 00:00:00| 113|              NULL|                NULL|3984.3200000000083|                  220|
|2020-01-01 00:00:00| 152|58

In [22]:
df_join.write.parquet('../../data/report/revenue/total', mode='overwrite')

26/03/10 13:15:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/10 13:15:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/10 13:15:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/10 13:15:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/10 13:15:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/10 13:15:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/10 13:15:30 WARN MemoryManager: Total allocation exceeds 95.

In [ ]:
# now instead of recomputing everything i want to use the data that i had already saved 

In [23]:
# Before we materialized the results of df_green_revenue / df_yellow_revenue

df_green_revenue = spark.read.parquet('../../data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('../../data/report/revenue/yellow')

In [24]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

df_join.write.parquet('../../data/report/revenue/total', mode='overwrite')

26/03/10 13:27:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/10 13:27:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/10 13:27:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/10 13:27:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/10 13:27:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/10 13:27:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/10 13:27:18 WARN MemoryManager: Total allocation exceeds 95.

In [ ]:
# -> in this case made no difference really for some reason

# Joining one large and one small table

In [25]:
df_join = spark.read.parquet('../../data/report/revenue/total')

In [26]:
df_join.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  22|              15.8|                   1|              NULL|                 NULL|
|2020-01-01 00:00:00|  29|              61.3|                   1|              NULL|                 NULL|
|2020-01-01 00:00:00|  40|            168.98|                   8|             89.97|                    5|
|2020-01-01 00:00:00|  81|54.870000000000005|                   2|             30.32|                    1|
|2020-01-01 00:00:00| 169|             64.17|                   3| 62.19999999999999|                    5|
|2020-01-01 00:00:00| 202|              NULL|                NULL|              10.3|                    1|
|2020-01-01 00:00:00| 233|  

In [27]:
# What exactly is zone? 

df_zones = spark.read.parquet('../setup_test/zones/')
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [33]:
# Now we want to join these two tables 

df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)


In [34]:
df_result.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+----------+-------------+--------------------+------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|      Borough|                Zone|service_zone|
+-------------------+----+------------------+--------------------+------------------+---------------------+----------+-------------+--------------------+------------+
|2020-01-01 00:00:00|  22|              15.8|                   1|              NULL|                 NULL|        22|     Brooklyn|    Bensonhurst West|   Boro Zone|
|2020-01-01 00:00:00|  29|              61.3|                   1|              NULL|                 NULL|        29|     Brooklyn|      Brighton Beach|   Boro Zone|
|2020-01-01 00:00:00|  40|            168.98|                   8|             89.97|                    5|        40|     Brooklyn|     Carroll Gardens|   Boro Zone

In [36]:
# we are writing it parquet because when we execute show it doesnt process the entire dataset but only a part of it
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

26/03/10 13:39:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/10 13:39:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/10 13:39:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/10 13:39:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/10 13:39:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/10 13:39:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/10 13:39:05 WARN MemoryManager: Total allocation exceeds 95.